### imports and hyperparams

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import random

# sequence length
horizon = 200

# model size
out_dim = 1
n_epochs = 150
feats = 1
tol = 5
units = 32


### load data

In [9]:
data  = np.load(
    "/home/michel/Documents/machineLearningForControl/code/gym-unbalanced-disk-5SC28-group-24/disc-benchmark-files/training-val-test-data.npz"
)
y_raw = data["th"]
u_raw = data["u"]


### make OE sequences

In [ ]:

def make_sequences(u, y, nf):
    # each sample: u[k-nf:k], y[k-nf:k]
    U, Y = [], []
    for k in range(nf, len(u) + 1):
        U.append(u[k-nf:k])
        Y.append(y[k-nf:k])
    return np.array(U), np.array(Y)


### split and convert to tensors

In [ ]:
# test set is the last 15%, no shuffle to keep time order
u_tv, u_test, y_tv, y_test = train_test_split(
    u_raw, y_raw, test_size=0.15, shuffle=False
)
u_train, u_val, y_train, y_val = train_test_split(
    u_tv, y_tv, test_size=0.40, shuffle=False
)

to_t = lambda seqs: [torch.tensor(s, dtype=torch.float32) for s in zip(*make_sequences(*seqs, nf=horizon))]

# build tensors
U_train_list, Y_train_list = make_sequences(u_train, y_train, horizon)
U_val, Y_val = make_sequences(u_val,   y_val,   horizon)
U_test, Y_test = make_sequences(u_test,  y_test,  horizon)

# shuffle training set
pairs = list(zip(U_train_list.tolist(), Y_train_list.tolist()))
random.shuffle(pairs)
u_shuf, y_shuf = zip(*pairs)

phi_train = torch.tensor(u_shuf, dtype=torch.float32)
t_train = torch.tensor(y_shuf, dtype=torch.float32)
phi_val = torch.tensor(U_val,  dtype=torch.float32)
t_val = torch.tensor(Y_val,  dtype=torch.float32)
phi_test = torch.tensor(U_test, dtype=torch.float32)
t_test = torch.tensor(Y_test, dtype=torch.float32)


### structure of RNN model

In [ ]:
class RNN(nn.Module):
    def __init__(self, n_feats, n_out, n_hidden):
        super().__init__()
        self.n_hidden = n_hidden

        # reuse same block for both transitions
        block = lambda n_in, n_out: nn.Sequential(
            nn.Linear(n_in, n_hidden), nn.Tanh(),
            nn.Linear(n_hidden, n_hidden), nn.Tanh(),
            nn.Linear(n_hidden, n_out),
        ).float()

        self.h2h = block(n_hidden + n_feats, n_hidden) # hidden state update
        self.h2o = block(n_hidden + n_feats, n_out) # output at each step

    def forward(self, x):
        h = torch.zeros(x.shape[0], self.n_hidden, dtype=torch.float32, device=x.device)
        outs = []
        for i in range(x.shape[1]):
            u = x[:, i].unsqueeze(1)
            cat = torch.cat([h, u], dim=1)
            outs.append(self.h2o(cat))
            h = self.h2h(cat)
        return torch.stack(outs, dim=1).squeeze(2)

# check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model     = RNN(feats, out_dim, units).to(device)
optimizer = torch.optim.Adam(model.parameters())
loss_fn   = nn.MSELoss()

# move data to dervice
phi_train, t_train = phi_train.to(device), t_train.to(device)
phi_val, t_val = phi_val.to(device), t_val.to(device)
phi_test, t_test = phi_test.to(device), t_test.to(device)
